# 5. Sum of the parts

**Question.** If divisions were valued on their own multiples, what is equity after one subtraction for parent net debt?

**Output.** Segment EV build, conglomerate discount, equity value per share, waterfall.

Live tickers need `SEGMENTS`. Without them, this notebook teaches on the sample company.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "valuation" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from the project directory.")

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataclasses import replace
import matplotlib.pyplot as plt
import pandas as pd

from valuation import boot
from valuation import plots as viz
from valuation import report as R
from valuation.case import TOL
from valuation.statements import Line, audit, combine, forecast_statements, identity_failures, max_abs_gap

# Compact equity picker — changes cases/active.yaml for EVERY notebook.
case, UNIT = boot(picker=True)


In [ ]:
from valuation import sample_case
from valuation.sotp import run_sotp

teaching = case
if not case.segments:
    R.note("No segments on this case — teaching on Northbridge. Set SEGMENTS for a live name.")
    teaching = sample_case()

forward = run_sotp(teaching, basis="forward")
latest = run_sotp(teaching, basis="ltm")
R.show_sotp(forward, teaching)
R.kpi_row([
    ("LTM price", f"${latest.price:,.2f}"),
    ("Forward price", f"${forward.price:,.2f}"),
    ("Entered price", f"{teaching.price}"),
])
viz.plot_sotp_waterfall(forward.by_segment, forward.net_debt, forward.equity_value, teaching.unit)
plt.show()

ops = forward.by_segment[~forward.by_segment["segment"].str.contains("discount|Corporate", case=False)]
viz.plot_composition(
    ops["segment"],
    ops["enterprise_value"].clip(lower=0),
    title="Segment EV mix (before corporate / discount)",
)
